In [7]:
import pandas as pd
import numpy as np
import boto3
from io import BytesIO

In [1]:
def load_parquet_from_s3(s3_client, bucket, key):
    response = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(response["Body"].read()))

In [5]:
# Init client
s3_client = boto3.client("s3")

In [8]:
# Load
df = load_parquet_from_s3(
    s3_client,
    bucket="swo-ngoctran-public",
    key="mlops/pipelines/01_preprocessing_kmeans/2024-01/30/23/sample_output1_DWA_processed.parquet"
)
df.shape

(1878424, 10)

In [9]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [14]:
# endpoint_name = "kmeans-speed-endpoint-20250708-011926"
endpoint_name = 'kmeans-speed-endpoint-20250708-025459'

In [15]:
# Tạo predictor gắn với endpoint
predictor = Predictor(
    endpoint_name=endpoint_name,
    serializer=CSVSerializer()
)

In [16]:

# Test với 1 giá trị từ cột "value" trong df
sample_value = df["value"].iloc[0]
response = predictor.predict(f"{sample_value}")

print("Predicted cluster:", response)

Predicted cluster: b'[2]'


In [17]:
# Lấy 5 sample
samples = df["value"].iloc[:5].tolist()

# Convert thành CSV (1 chiều)
csv_payload = "\n".join([str(x) for x in samples])
response = predictor.predict(csv_payload)

print("Predicted clusters:", response)


Predicted clusters: b'[2, 2, 2, 2, 2]'
